# Composite DNA Dataset Generator — Uniform-Ratio Alphabets

**Single run can generate one profile or all eight.** Set `ERROR_MODEL` in Cell 2 to either:
- a single profile name (e.g. `"EZ17"`) → generates one dataset, or
- a list of profile names (e.g. `["EZ17", "R21"]`) → generates each in turn, or
- the string `"ALL"` → generates all eight profiles in order.

Existing `.pkl` files are skipped by default; set `OVERWRITE = True` in Cell 9 to force regeneration. Within a batch run, each profile is independently seeded from the same base `SEED` so individual profile datasets are byte-for-byte reproducible regardless of whether you generate them one at a time or in a batch.

**Supports 3 alphabet modes:** `2mix_only` (10 classes), `2mix_3mix` (14 classes), `2mix_3mix_4mix` (15 classes).

**Supports all 8 error profiles** from Bar-Lev *et al.*, *Nature Machine Intelligence* (2025) and its supplementary material:

| Profile | Platform | n | Source |
|---|---|---|---|
| EZ17  | Illumina miSeq + Twist        | 136 | Erlich & Zielinski 2017 (main paper Table:ErrorRates) |
| G15   | Illumina miSeq + CustomArray  | 104 | Grass *et al.* 2015 |
| O17   | Illumina NextSeq + Twist      | 77  | Organick *et al.* 2018 |
| R21   | Nanopore MinION + Twist       | 136 | Bar-Lev SDG (supplementary Table:Supp_NewErrorRates) |
| B22   | Nanopore MinION-short + Twist | 136 | Bar-Lev SDG |
| BOS22 | Illumina miSeq 2022 + Twist   | 136 | Bar-Lev SDG |
| NP22  | Nanopore Pilot Nov-2022       | 136 | Bar-Lev SDG |
| NPF22 | Nanopore Full Nov-2022        | 136 | Bar-Lev SDG |

All per-base error rates are transcribed **verbatim** from `other_github_dataset` in project memory (the SDG of Bar-Lev *et al.*).
The insertion event probability uses the per-base **`pi`** (pre-insertion) column, matching the github simulator's `generate_error_type_for_base` semantics. Long deletions are applied with length 2–6 drawn from the github `long_deletion_length_rates` distribution.

In [1]:
# =============================================================================
# CELL 1: IMPORTS
# =============================================================================
import random
import numpy as np
import pickle
import os
import time
from collections import Counter
from datetime import datetime

In [2]:
# =============================================================================
# CELL 2: CONFIGURATION  (single profile OR batch over all 8 profiles)
# =============================================================================

# ------------------- SELECT ERROR MODEL(S) -------------------
# Three ways to set this:
#   1. Single profile: e.g. "EZ17"  → generates ONE dataset.
#   2. List of profiles: e.g. ["EZ17", "R21"]  → generates those datasets in order.
#   3. The string "ALL" → generates ALL 8 profiles in order.
# Valid profile names: "EZ17", "G15", "O17", "R21", "B22", "BOS22", "NP22", "NPF22"
# Backwards-compat aliases also accepted: "erlich"=EZ17, "grass"=G15, "organick"=O17
ERROR_MODEL = "ALL"  # <-- CHANGE TO "ALL" FOR BATCH MODE

# ------------------- SELECT ALPHABET MODE -------------------
# Options: "2mix_only", "2mix_3mix", "2mix_3mix_4mix"
ALPHABET_MODE = "2mix_only"  # <-- CHANGE THIS TO SELECT ALPHABET MODE

# ------------------- GLOBAL DATASET PARAMETERS -------------------
# These apply to every profile generated in this run.
NUM_SAMPLES  = 100000
MAX_COVERAGE = 50
SEED         = 42
DATASET_DIR  = "./dataset"


# All 8 profiles supported by this notebook.
# Sequence lengths and oligo design are from the paper (main + supplementary).
ERROR_MODEL_SPECS = {
    # --- main-paper profiles ---
    "EZ17":  {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "EZ17",  "platform": "Illumina miSeq + Twist Bioscience",
              "reference": "Erlich & Zielinski 2017"},
    "G15":   {"full_length": 117, "index_length": 13, "seq_length": 104,
              "name": "G15",   "platform": "Illumina miSeq + CustomArray",
              "reference": "Grass et al. 2015"},
    "O17":   {"full_length": 110, "index_length": 33, "seq_length": 77,
              "name": "O17",   "platform": "Illumina NextSeq + Twist Bioscience",
              "reference": "Organick et al. 2018"},
    # --- supplementary profiles (same 152nt Twist pool re-sequenced) ---
    "R21":   {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "R21",   "platform": "Oxford Nanopore MinION + Twist Bioscience",
              "reference": "Bar-Lev et al. 2024 (SDG)"},
    "B22":   {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "B22",   "platform": "Nanopore MinION short-read + Twist Bioscience",
              "reference": "Bar-Lev et al. 2024 (SDG)"},
    "BOS22": {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "BOS22", "platform": "Illumina miSeq 2022 + Twist Bioscience",
              "reference": "Bar-Lev & Sabary 2022"},
    "NP22":  {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "NP22",  "platform": "Nanopore Pilot Nov-2022 + Twist Bioscience",
              "reference": "Bar-Lev & Sabary 2022"},
    "NPF22": {"full_length": 152, "index_length": 16, "seq_length": 136,
              "name": "NPF22", "platform": "Nanopore Full Pool Nov-2022 + Twist Bioscience",
              "reference": "Bar-Lev & Sabary 2022"},
}

# Aliases (for backwards compatibility with previous scripts)
ERROR_MODEL_ALIASES = {
    "erlich": "EZ17", "grass": "G15", "organick": "O17",
}
def resolve_error_model(name):
    return ERROR_MODEL_ALIASES.get(name, name)

VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15,
}

# ------------------- Resolve ERROR_MODEL into a list of profiles -------------------
ALL_PROFILES = ['EZ17', 'G15', 'O17', 'R21', 'B22', 'BOS22', 'NP22', 'NPF22']

def _resolve_to_list(spec):
    if isinstance(spec, str):
        if spec.upper() == 'ALL':
            return list(ALL_PROFILES)
        return [resolve_error_model(spec)]
    elif isinstance(spec, (list, tuple)):
        return [resolve_error_model(s) for s in spec]
    else:
        raise TypeError(f'ERROR_MODEL must be str or list, got {type(spec)}')

PROFILES_TO_RUN = _resolve_to_list(ERROR_MODEL)
for p in PROFILES_TO_RUN:
    assert p in ERROR_MODEL_SPECS, (
        f'Unknown profile {p}. Valid: {list(ERROR_MODEL_SPECS.keys())}'
    )

os.makedirs(DATASET_DIR, exist_ok=True)

def make_config_for_profile(profile_name, alphabet_mode):
    """Build a per-profile CONFIG dict from the global parameters."""
    specs = ERROR_MODEL_SPECS[profile_name]
    cfg = {
        'error_model': profile_name,
        'error_specs': specs,
        'alphabet_mode': alphabet_mode,
        'num_samples': NUM_SAMPLES,
        'seq_length': specs['seq_length'],
        'max_coverage': MAX_COVERAGE,
        'dataset_dir': DATASET_DIR,
        'seed': SEED,
        'vocab_size': VOCAB_SIZES[alphabet_mode],
    }
    name = f"dna_{specs['name']}_{alphabet_mode}"
    cfg['dataset_path'] = f"{DATASET_DIR}/{name}_{NUM_SAMPLES}_{MAX_COVERAGE}.pkl"
    return cfg

print("=" * 72)
print("DATASET GENERATION CONFIGURATION")
print("=" * 72)
if len(PROFILES_TO_RUN) == 1:
    print(f'   Mode           : SINGLE profile')
else:
    print(f'   Mode           : BATCH ({len(PROFILES_TO_RUN)} profiles)')
print(f"   Profiles       : {', '.join(PROFILES_TO_RUN)}")
print(f"   Alphabet Mode  : {ALPHABET_MODE}")
print(f"   Vocab Size     : {VOCAB_SIZES[ALPHABET_MODE]} classes")
print(f"   Num Samples    : {NUM_SAMPLES:,}")
print(f"   Max Coverage   : {MAX_COVERAGE}")
print(f"   Seed           : {SEED}")
print(f"   Output dir     : {DATASET_DIR}/")
print()
print(f"   Planned outputs:")
for p in PROFILES_TO_RUN:
    cfg = make_config_for_profile(p, ALPHABET_MODE)
    n = cfg['seq_length']
    print(f"     • {p:<6} (n={n})  →  {cfg['dataset_path']}")
print("=" * 72)

DATASET GENERATION CONFIGURATION
   Mode           : BATCH (8 profiles)
   Profiles       : EZ17, G15, O17, R21, B22, BOS22, NP22, NPF22
   Alphabet Mode  : 2mix_only
   Vocab Size     : 10 classes
   Num Samples    : 100,000
   Max Coverage   : 50
   Seed           : 42
   Output dir     : ./dataset/

   Planned outputs:
     • EZ17   (n=136)  →  ./dataset/dna_EZ17_2mix_only_100000_50.pkl
     • G15    (n=104)  →  ./dataset/dna_G15_2mix_only_100000_50.pkl
     • O17    (n=77)  →  ./dataset/dna_O17_2mix_only_100000_50.pkl
     • R21    (n=136)  →  ./dataset/dna_R21_2mix_only_100000_50.pkl
     • B22    (n=136)  →  ./dataset/dna_B22_2mix_only_100000_50.pkl
     • BOS22  (n=136)  →  ./dataset/dna_BOS22_2mix_only_100000_50.pkl
     • NP22   (n=136)  →  ./dataset/dna_NP22_2mix_only_100000_50.pkl
     • NPF22  (n=136)  →  ./dataset/dna_NPF22_2mix_only_100000_50.pkl


In [3]:
# =============================================================================
# CELL 3: SEED HELPER
# =============================================================================
# In batch mode, set_seed() is called once per profile so that each profile's
# clean labels and noise realisations are reproducible from the same SEED.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

print(f'Seed helper ready. Base seed = {SEED}.')

Seed helper ready. Base seed = 42.


In [4]:
# =============================================================================
# CELL 4: COMPOSITE DNA ALPHABET DEFINITIONS (uniform-ratio mixtures)
# =============================================================================

PURE_BASES = {
    'A': ['A'], 'C': ['C'], 'G': ['G'], 'T': ['T'],
}

# All C(4,2) = 6 two-nucleotide mixtures, 0.5/0.5 uniform.
TWO_MIX_MAP = {
    'M1': ['A', 'T'],  # A|T
    'M2': ['C', 'G'],  # C|G
    'M3': ['C', 'T'],  # C|T
    'M4': ['G', 'T'],  # G|T
    'M5': ['A', 'C'],  # A|C
    'M6': ['A', 'G'],  # A|G
}

# All C(4,3) = 4 three-nucleotide mixtures, 1/3 each.
THREE_MIX_MAP = {
    'T1': ['A', 'C', 'G'],
    'T2': ['A', 'C', 'T'],
    'T3': ['A', 'G', 'T'],
    'T4': ['C', 'G', 'T'],
}

# The single four-nucleotide mixture, 0.25 each.
FOUR_MIX_MAP = {
    'Q1': ['A', 'C', 'G', 'T'],
}

def build_composite_map(mode):
    m = dict(PURE_BASES)
    m.update(TWO_MIX_MAP)
    if mode in ('2mix_3mix', '2mix_3mix_4mix'):
        m.update(THREE_MIX_MAP)
    if mode == '2mix_3mix_4mix':
        m.update(FOUR_MIX_MAP)
    return m

def build_symbol_to_idx(mode):
    idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3,
           'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9}
    if mode in ('2mix_3mix', '2mix_3mix_4mix'):
        idx.update({'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13})
    if mode == '2mix_3mix_4mix':
        idx.update({'Q1': 14})
    return idx

def build_ideal_vectors(mode):
    vecs = [
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        [0.5, 0.0, 0.0, 0.5],  # M1 A|T
        [0.0, 0.5, 0.5, 0.0],  # M2 C|G
        [0.0, 0.5, 0.0, 0.5],  # M3 C|T
        [0.0, 0.0, 0.5, 0.5],  # M4 G|T
        [0.5, 0.5, 0.0, 0.0],  # M5 A|C
        [0.5, 0.0, 0.5, 0.0],  # M6 A|G
    ]
    if mode in ('2mix_3mix', '2mix_3mix_4mix'):
        t = 1.0/3.0
        vecs.extend([
            [t, t, t, 0.0],  # T1 A|C|G
            [t, t, 0.0, t],  # T2 A|C|T
            [t, 0.0, t, t],  # T3 A|G|T
            [0.0, t, t, t],  # T4 C|G|T
        ])
    if mode == '2mix_3mix_4mix':
        vecs.append([0.25, 0.25, 0.25, 0.25])  # Q1 A|C|G|T
    return vecs

COMPOSITE_MAP  = build_composite_map(ALPHABET_MODE)
SYMBOL_TO_IDX  = build_symbol_to_idx(ALPHABET_MODE)
IDEAL_VECTORS  = build_ideal_vectors(ALPHABET_MODE)
ALL_SYMBOLS    = list(COMPOSITE_MAP.keys())

print(f"\nComposite alphabet ({ALPHABET_MODE}):")
print(f"   Total symbols: {len(ALL_SYMBOLS)}")
print(f"\n   {'Symbol':<8} {'Index':<6} {'Composition':<15} Ideal Vector [A,C,G,T]")
print(f"   {'-'*60}")
for sym in ALL_SYMBOLS:
    i = SYMBOL_TO_IDX[sym]
    comp = ' | '.join(COMPOSITE_MAP[sym])
    v = IDEAL_VECTORS[i]
    print(f"   {sym:<8} {i:<6} {comp:<15} [{v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}, {v[3]:.3f}]")


Composite alphabet (2mix_only):
   Total symbols: 10

   Symbol   Index  Composition     Ideal Vector [A,C,G,T]
   ------------------------------------------------------------
   A        0      A               [1.000, 0.000, 0.000, 0.000]
   C        1      C               [0.000, 1.000, 0.000, 0.000]
   G        2      G               [0.000, 0.000, 1.000, 0.000]
   T        3      T               [0.000, 0.000, 0.000, 1.000]
   M1       4      A | T           [0.500, 0.000, 0.000, 0.500]
   M2       5      C | G           [0.000, 0.500, 0.500, 0.000]
   M3       6      C | T           [0.000, 0.500, 0.000, 0.500]
   M4       7      G | T           [0.000, 0.000, 0.500, 0.500]
   M5       8      A | C           [0.500, 0.500, 0.000, 0.000]
   M6       9      A | G           [0.500, 0.000, 0.500, 0.000]


In [5]:
# =============================================================================
# CELL 5: ERROR-RATES CLASS — all 8 profiles, verbatim from Bar-Lev SDG
# =============================================================================

class ErrorRates:
    """
    Error-rate configuration for the 8 sequencing/synthesis profiles used in
    Bar-Lev et al., "Scalable and robust DNA-based storage via coding theory
    and deep learning", Nature Machine Intelligence (2025), and characterised
    via SOLQC.

    Per-base values are transcribed verbatim from the Synthetic Data Generator
    (SDG) at https://github.com/itaiorr/Deep-DNA-based-storage  (the
    ErrorRates and ClusterGenerator classes).

    Storage convention per base:
        's'  : substitution probability
        'i'  : "which-base-to-insert" weight (used to pick the inserted symbol)
        'pi' : pre-insertion probability  -> EVENT probability of an insertion
        'd'  : single-base deletion probability
        'ld' : long-deletion event probability (length drawn from
               long_deletion_length_rates below)

    Profiles & oligo design (all from Bar-Lev SDG / paper Table:ErrorRates &
    supplementary Table:Supp_NewErrorRates):

        EZ17  : Illumina miSeq    + Twist        152nt, 16nt index -> n=136
        G15   : Illumina miSeq    + CustomArray  117nt, 13nt index -> n=104
        O17   : Illumina NextSeq  + Twist        110nt, 33nt index -> n=77
        R21   : Nanopore MinION   + Twist        152nt, 16nt index -> n=136
        B22   : Nanopore MinION-S + Twist        152nt, 16nt index -> n=136
        BOS22 : Illumina miSeq'22 + Twist        152nt, 16nt index -> n=136
        NP22  : Nanopore Pilot    + Twist        152nt, 16nt index -> n=136
        NPF22 : Nanopore Full     + Twist        152nt, 16nt index -> n=136
    """

    # Long-deletion length distribution (from github ClusterGenerator).
    LONG_DELETION_LENGTH_RATES = {
        2: 2.8e-4,
        3: 7.75e-5,
        4: 3.25e-5,
        5: 1.0e-6,
        6: 5.5e-8,
    }

    def __init__(self):
        self.general_errors = {'d': 0.0, 'ld': 0.0, 'i': 0.0, 's': 0.0}
        self.per_base_errors = {
            'A': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'C': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'G': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
            'T': {'s': 0.0, 'i': 0.0, 'pi': 0.0, 'd': 0.0, 'ld': 0.0},
        }
        self.profile_name = None
        self.long_deletion_length_rates = dict(self.LONG_DELETION_LENGTH_RATES)

    # ---------------------------------------------------------------------
    # ORIGINAL 3 PROFILES (main paper Table:ErrorRates)
    # ---------------------------------------------------------------------

    def set_EZ17_values(self):
        """Erlich & Zielinski 2017 (Illumina miSeq + Twist Bioscience)."""
        self.profile_name = "EZ17"
        self.general_errors = {'s': 1.32e-03, 'i': 5.81e-04, 'd': 9.58e-04, 'ld': 2.33e-04}
        self.per_base_errors['A'] = {'s': 0.00135, 'i': 0.00057, 'pi': 0.00059, 'd': 0.00099, 'ld': 0.00024}
        self.per_base_errors['C'] = {'s': 0.00135, 'i': 0.00059, 'pi': 0.00058, 'd': 0.00098, 'ld': 0.00023}
        self.per_base_errors['G'] = {'s': 0.00126, 'i': 0.00059, 'pi': 0.00057, 'd': 0.00094, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00132, 'i': 0.00058, 'pi': 0.00058, 'd': 0.00096, 'ld': 0.00023}

    def set_G15_values(self):
        """Grass et al. 2015 (Illumina miSeq + CustomArray)."""
        self.profile_name = "G15"
        self.general_errors = {'s': 5.84e-03, 'i': 8.57e-04, 'd': 5.37e-03, 'ld': 3.48e-04}
        self.per_base_errors['A'] = {'s': 0.00605, 'i': 0.00090, 'pi': 0.00092, 'd': 0.00543, 'ld': 0.00036}
        self.per_base_errors['C'] = {'s': 0.00563, 'i': 0.00083, 'pi': 0.00081, 'd': 0.00513, 'ld': 0.00034}
        self.per_base_errors['G'] = {'s': 0.00577, 'i': 0.00085, 'pi': 0.00087, 'd': 0.00539, 'ld': 0.00034}
        self.per_base_errors['T'] = {'s': 0.00591, 'i': 0.00084, 'pi': 0.00084, 'd': 0.00559, 'ld': 0.00036}

    def set_O17_values(self):
        """Organick et al. 2018 (Illumina NextSeq + Twist Bioscience)."""
        self.profile_name = "O17"
        self.general_errors = {'s': 2.52e-03, 'i': 4.14e-04, 'd': 6.94e-04, 'ld': 2.11e-04}
        self.per_base_errors['A'] = {'s': 0.00717, 'i': 0.00030, 'pi': 0.00120, 'd': 0.00201, 'ld': 0.00054}
        self.per_base_errors['C'] = {'s': 0.00034, 'i': 0.00007, 'pi': 0.00007, 'd': 0.00006, 'ld': 0.00001}
        self.per_base_errors['G'] = {'s': 0.00196, 'i': 0.00125, 'pi': 0.00029, 'd': 0.00058, 'ld': 0.00023}
        self.per_base_errors['T'] = {'s': 0.00055, 'i': 0.00006, 'pi': 0.00009, 'd': 0.00014, 'ld': 0.00006}

    # ---------------------------------------------------------------------
    # FIVE SUPPLEMENTARY PROFILES (supplementary Table:Supp_NewErrorRates)
    # ---------------------------------------------------------------------

    def set_R21_values(self):
        """Rang/Bar-Lev 2021 (Nanopore MinION + Twist Bioscience)."""
        self.profile_name = "R21"
        self.general_errors = {'s': 1.08e-02, 'i': 1.65e-02, 'd': 1.18e-02, 'ld': 3.36e-03}
        self.per_base_errors['A'] = {'s': 1.039e-02, 'i': 1.610e-02, 'pi': 1.585e-02, 'd': 1.192e-02, 'ld': 0.314e-02}
        self.per_base_errors['C'] = {'s': 1.042e-02, 'i': 1.639e-02, 'pi': 1.578e-02, 'd': 1.260e-02, 'ld': 0.334e-02}
        self.per_base_errors['G'] = {'s': 1.094e-02, 'i': 1.604e-02, 'pi': 1.700e-02, 'd': 1.267e-02, 'ld': 0.337e-02}
        self.per_base_errors['T'] = {'s': 1.131e-02, 'i': 1.738e-02, 'pi': 1.729e-02, 'd': 1.327e-02, 'ld': 0.357e-02}

    def set_B22_values(self):
        """Bar-Lev 2022 (Nanopore MinION short-read + Twist Bioscience)."""
        self.profile_name = "B22"
        self.general_errors = {'s': 1.12e-02, 'i': 1.08e-02, 'd': 7.87e-03, 'ld': 2.05e-03}
        self.per_base_errors['A'] = {'s': 0.01146, 'i': 0.01104, 'pi': 0.01096, 'd': 0.00798, 'ld': 0.00210}
        self.per_base_errors['C'] = {'s': 0.01117, 'i': 0.01092, 'pi': 0.01088, 'd': 0.00804, 'ld': 0.00208}
        self.per_base_errors['G'] = {'s': 0.01129, 'i': 0.01073, 'pi': 0.01071, 'd': 0.00792, 'ld': 0.00201}
        self.per_base_errors['T'] = {'s': 0.01094, 'i': 0.01039, 'pi': 0.01053, 'd': 0.00769, 'ld': 0.00201}

    def set_BOS22_values(self):
        """Bar-Lev & Sabary 2022 (Illumina miSeq 2022 + Twist Bioscience)."""
        self.profile_name = "BOS22"
        self.general_errors = {'s': 5.29e-04, 'i': 5.42e-05, 'd': 7.08e-05, 'ld': 1.81e-05}
        self.per_base_errors['A'] = {'s': 0.004e-2, 'i': 0.004e-2, 'pi': 0.0,      'd': 0.0,      'ld': 0.0}
        self.per_base_errors['C'] = {'s': 0.024e-2, 'i': 0.006e-2, 'pi': 0.002e-2, 'd': 0.003e-2, 'ld': 0.001e-2}
        self.per_base_errors['G'] = {'s': 0.004e-2, 'i': 0.005e-2, 'pi': 0.0,      'd': 0.0,      'ld': 0.0}
        self.per_base_errors['T'] = {'s': 0.175e-2, 'i': 0.007e-2, 'pi': 0.018e-2, 'd': 0.024e-2, 'ld': 0.006e-2}

    def set_NP22_values(self):
        """Bar-Lev & Sabary 2022 (Nanopore Pilot Pool Nov-2022 + Twist).
        github name: BOS22PILOTOMER."""
        self.profile_name = "NP22"
        self.general_errors = {'s': 1.29e-02, 'i': 1.16e-02, 'd': 9.75e-03, 'ld': 2.82e-03}
        self.per_base_errors['A'] = {'s': 0.520e-2, 'i': 0.644e-2, 'pi': 0.582e-2, 'd': 0.373e-2, 'ld': 0.098e-2}
        self.per_base_errors['C'] = {'s': 1.498e-2, 'i': 1.331e-2, 'pi': 1.340e-2, 'd': 1.178e-2, 'ld': 0.336e-2}
        self.per_base_errors['G'] = {'s': 0.638e-2, 'i': 0.735e-2, 'pi': 0.667e-2, 'd': 0.474e-2, 'ld': 0.128e-2}
        self.per_base_errors['T'] = {'s': 2.437e-2, 'i': 1.867e-2, 'pi': 1.983e-2, 'd': 1.903e-2, 'ld': 0.550e-2}

    def set_NPF22_values(self):
        """Bar-Lev & Sabary 2022 (Nanopore Full Pool Nov-2022 + Twist).
        github name: full_dataset (noise_coef=1)."""
        self.profile_name = "NPF22"
        self.general_errors = {'s': 1.56e-02, 'i': 1.24e-02, 'd': 9.79e-03, 'ld': 2.67e-03}
        self.per_base_errors['A'] = {'s': 1.700e-2, 'i': 1.320e-2, 'pi': 1.343e-2, 'd': 1.095e-2, 'ld': 0.294e-2}
        self.per_base_errors['C'] = {'s': 1.606e-2, 'i': 1.271e-2, 'pi': 1.272e-2, 'd': 1.038e-2, 'ld': 0.277e-2}
        self.per_base_errors['G'] = {'s': 1.593e-2, 'i': 1.256e-2, 'pi': 1.255e-2, 'd': 1.014e-2, 'ld': 0.273e-2}
        self.per_base_errors['T'] = {'s': 1.358e-2, 'i': 1.116e-2, 'pi': 1.094e-2, 'd': 0.849e-2, 'ld': 0.224e-2}

    # ---------------------------------------------------------------------
    # DISPATCHER
    # ---------------------------------------------------------------------
    def set_values_by_model(self, model_name):
        dispatch = {
            "erlich":   self.set_EZ17_values,  "EZ17":  self.set_EZ17_values,
            "grass":    self.set_G15_values,   "G15":   self.set_G15_values,
            "organick": self.set_O17_values,   "O17":   self.set_O17_values,
            "R21":      self.set_R21_values,
            "B22":      self.set_B22_values,
            "BOS22":    self.set_BOS22_values,
            "NP22":     self.set_NP22_values,
            "NPF22":    self.set_NPF22_values,
        }
        if model_name not in dispatch:
            raise ValueError(f"Unknown error model: {model_name}. "
                             f"Valid: {sorted(dispatch.keys())}")
        dispatch[model_name]()

    def print_current_values(self):
        print(f"   Profile: {self.profile_name}")
        g = self.general_errors
        print(f"   General: s={g['s']:.5e}  i={g['i']:.5e}  d={g['d']:.5e}  ld={g['ld']:.5e}")
        for b in 'ACGT':
            r = self.per_base_errors[b]
            print(f"   {b}: s={r['s']:.5e}  pi(ins)={r['pi']:.5e}  d={r['d']:.5e}  ld={r['ld']:.5e}  i(weight)={r['i']:.5e}")


In [6]:
# =============================================================================
# CELL 6: SEQUENCE GENERATION FUNCTIONS
# =============================================================================

def generate_composite_sequence(length, composite_map):
    symbols = list(composite_map.keys())
    return [random.choice(symbols) for _ in range(length)]

def realize_sequence(composite_seq, composite_map):
    """Realise composite symbols into a single concrete DNA sequence by
    drawing each position uniformly from its component nucleotides."""
    out = []
    for sym in composite_seq:
        out.append(random.choice(composite_map[sym]))
    return ''.join(out)

def apply_ids_noise(sequence, error_profile):
    """
    Two-stage IDS + long-deletion noise injector, matching
    cluster_generator.NoisyStrandGenerator from the Bar-Lev SDG exactly.

    STAGE 1: outer error-vs-no-error coin uses GENERAL rates.
    STAGE 2: conditional on error, type is drawn with PER-BASE weights
             over {s, pi, d, ld}.

    On an insertion event the inserted base is emitted BEFORE the original
    base, the original base is then emitted unchanged, and processing
    advances past it (the original base is consumed -- it does not receive
    a further error). This reproduces the github index arithmetic exactly.
    """
    bases = ['A', 'C', 'G', 'T']

    # ---- STAGE 1 probability: total error rate from GENERAL rates ----
    g = error_profile.general_errors
    p_total_err = g['s'] + g['i'] + g['d'] + g['ld']
    if p_total_err > 1.0:
        p_total_err = 1.0

    # ---- Long-deletion length distribution ----
    ld_lengths = list(error_profile.long_deletion_length_rates.keys())
    ld_weights = list(error_profile.long_deletion_length_rates.values())

    # ---- Per-base weights for choosing which base to INSERT ----
    # (matches inject_insertion: weights come from per_base[*]['i'])
    ins_base_weights = [error_profile.per_base_errors[x]['i'] for x in bases]
    if sum(ins_base_weights) <= 0:
        ins_base_weights = [1.0, 1.0, 1.0, 1.0]  # safety fallback

    out_chars = []
    n = len(sequence)
    pos = 0
    while pos < n:
        b = sequence[pos]
        if b not in bases:
            out_chars.append(b)
            pos += 1
            continue

        # STAGE 1: outer error-vs-no-error coin (general rates)
        if random.random() >= p_total_err:
            out_chars.append(b)
            pos += 1
            continue

        # STAGE 2: choose event type using PER-BASE weights {s, pi, d, ld}
        # (matches github generate_error_type_for_base which deletes 'i')
        pbr = error_profile.per_base_errors[b]
        event_types   = ['s', 'pi', 'd', 'ld']
        event_weights = [pbr['s'], pbr['pi'], pbr['d'], pbr['ld']]
        if sum(event_weights) <= 0:
            # numerical degenerate: treat as no-error
            out_chars.append(b)
            pos += 1
            continue
        event = random.choices(event_types, weights=event_weights, k=1)[0]

        if event == 's':
            # substitution: uniformly random base != b
            opts = [x for x in bases if x != b]
            out_chars.append(random.choice(opts))
            pos += 1
        elif event == 'pi':
            # insertion: pick which base to insert via per-base 'i' weights,
            # emit it BEFORE the original base, then emit the original base
            # unchanged and ADVANCE past it. The original base is consumed
            # and receives no further error (matches github inject_insertion
            # net index behaviour; verified by instrumented trace).
            ins_b = random.choices(bases, weights=ins_base_weights, k=1)[0]
            out_chars.append(ins_b)
            out_chars.append(b)
            pos += 1
        elif event == 'd':
            # single-base deletion: drop b
            pos += 1
        elif event == 'ld':
            # long deletion of length L starting at current position
            L = random.choices(ld_lengths, weights=ld_weights, k=1)[0]
            pos += L

    return "".join(out_chars)


In [7]:
# =============================================================================
# CELL 7: MAIN DATASET GENERATION FUNCTION
# =============================================================================

def generate_dataset(config, composite_map, symbol_to_idx, ideal_vectors):
    errors = ErrorRates()
    errors.set_values_by_model(config['error_model'])
    errors.print_current_values()

    num_samples = config['num_samples']
    seq_length  = config['seq_length']
    coverage    = config['max_coverage']
    filename    = config['dataset_path']

    dataset = {
        'metadata': {
            'type': f'Composite DNA ({config["alphabet_mode"]})',
            'error_profile': config['error_specs']['name'],
            'platform': config['error_specs']['platform'],
            'num_samples': num_samples,
            'seq_length': seq_length,
            'coverage_depth': coverage,
            'vocab_size': config['vocab_size'],
            'alphabet_mode': config['alphabet_mode'],
            'symbols': list(symbol_to_idx.keys()),
            'symbol_to_idx': symbol_to_idx,
            'ideal_vectors': ideal_vectors,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'seed': config['seed'],
        },
        'data': [],
    }

    print(f"\n{'='*64}")
    print("GENERATING DATASET")
    print(f"{'='*64}")
    print(f"   Samples       : {num_samples:,}")
    print(f"   Seq length    : {seq_length}")
    print(f"   Coverage depth: {coverage}")
    print(f"   Alphabet      : {config['alphabet_mode']} ({config['vocab_size']} classes)")
    print(f"{'='*64}")

    t0 = time.time()
    for i in range(num_samples):
        clean = generate_composite_sequence(seq_length, composite_map)
        reads = []
        for _ in range(coverage):
            realized = realize_sequence(clean, composite_map)
            reads.append(apply_ids_noise(realized, errors))
        dataset['data'].append({'id': i, 'label': clean, 'cluster': reads})
        if (i + 1) % 10000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta_s = (num_samples - i - 1) / rate
            print(f'   Processed {i+1:,}/{num_samples:,} | {rate:.1f} samples/s | ETA {eta_s:.1f}s')

    with open(filename, 'wb') as f:
        pickle.dump(dataset, f)

    total = time.time() - t0
    print(f"\n{'='*64}")
    print('DATASET GENERATION COMPLETE')
    print(f"{'='*64}")
    print(f'   Output file : {filename}')
    print(f'   Total time  : {total:.1f}s  ({total/60:.1f} min)')
    print(f'   File size   : {os.path.getsize(filename) / (1024*1024):.1f} MB')
    print(f"\n   Sample 0:")
    print(f"     Label (first 10): {dataset['data'][0]['label'][:10]}")
    print(f"     Read 1 (first 20): {dataset['data'][0]['cluster'][0][:20]}...")

    return dataset

In [8]:
# =============================================================================
# CELL 8: ANALYZE DATASET STATISTICS
# =============================================================================

def analyze_dataset(dataset, symbol_to_idx):
    print(f"\n{'='*64}")
    print('DATASET STATISTICS')
    print(f"{'='*64}")
    all_syms = []
    for s in dataset['data']:
        all_syms.extend(s['label'])
    counter = Counter(all_syms)
    total = len(all_syms)
    print(f"\nSymbol distribution in labels:")
    print(f"   {'Symbol':<8} {'Count':>10} {'%':>10} {'Expected':>10}")
    expected = 100.0 / len(symbol_to_idx)
    for sym in sorted(counter.keys(), key=lambda x: symbol_to_idx[x]):
        c = counter[sym]
        print(f"   {sym:<8} {c:>10,} {100*c/total:>9.2f}% {expected:>9.2f}%")
    print(f"   {'Total':<8} {total:>10,}")

    read_lens = [len(r) for s in dataset['data'] for r in s['cluster']]
    print(f"\nRead length statistics:")
    print(f"   Original length : {dataset['metadata']['seq_length']}")
    print(f"   Mean read length: {np.mean(read_lens):.2f}")
    print(f"   Std  read length: {np.std(read_lens):.2f}")
    print(f"   Min  / Max      : {np.min(read_lens)} / {np.max(read_lens)}")

In [9]:
# =============================================================================
# CELL 9: MAIN EXECUTION  (single profile OR batch loop)
# =============================================================================

# Control whether to overwrite existing datasets.
# Set to True to force regeneration even if the .pkl already exists.
OVERWRITE = False

results_summary = []
t_batch_start = time.time()

for run_idx, profile_name in enumerate(PROFILES_TO_RUN, start=1):
    cfg = make_config_for_profile(profile_name, ALPHABET_MODE)

    print(f"\n\n{'#'*72}")
    print(f"#  [{run_idx}/{len(PROFILES_TO_RUN)}]  Profile: {profile_name}  ({cfg['error_specs']['platform']})")
    print(f"#  n={cfg['seq_length']}  alphabet={ALPHABET_MODE}({cfg['vocab_size']})  N={NUM_SAMPLES:,}  M={MAX_COVERAGE}")
    print(f"{'#'*72}")

    if (not OVERWRITE) and os.path.exists(cfg['dataset_path']):
        size_mb = os.path.getsize(cfg['dataset_path']) / (1024*1024)
        print(f"[skip] Dataset already exists ({size_mb:.1f} MB): {cfg['dataset_path']}")
        print('       Set OVERWRITE = True to force regeneration.')
        results_summary.append({'profile': profile_name, 'status': 'skipped (exists)',
                                'path': cfg['dataset_path'], 'time_s': 0.0})
        continue

    # Re-seed once per profile so each profile is reproducible from base SEED.
    set_seed(SEED)

    t0 = time.time()
    try:
        dataset = generate_dataset(cfg, COMPOSITE_MAP, SYMBOL_TO_IDX, IDEAL_VECTORS)
        analyze_dataset(dataset, SYMBOL_TO_IDX)
        dt = time.time() - t0
        results_summary.append({'profile': profile_name, 'status': 'ok',
                                'path': cfg['dataset_path'], 'time_s': dt})
    except Exception as e:
        dt = time.time() - t0
        print(f'\n[ERROR] Profile {profile_name} failed: {type(e).__name__}: {e}')
        results_summary.append({'profile': profile_name, 'status': f'FAILED: {e}',
                                'path': cfg['dataset_path'], 'time_s': dt})
        # continue to the next profile rather than aborting the whole batch
        continue

# ---- Final batch summary ----
t_batch = time.time() - t_batch_start
print(f"\n\n{'='*72}")
print('BATCH SUMMARY')
print(f"{'='*72}")
print(f"   {'Profile':<8} {'Status':<22} {'Time':>10}   File")
print(f"   {'-'*8} {'-'*22} {'-'*10}   {'-'*40}")
for r in results_summary:
    t_s = f"{r['time_s']:.1f}s" if r['time_s'] > 0 else '—'
    print(f"   {r['profile']:<8} {r['status']:<22} {t_s:>10}   {r['path']}")
print(f"   {'-'*72}")
n_ok = sum(1 for r in results_summary if r['status'] == 'ok')
n_skip = sum(1 for r in results_summary if 'skipped' in r['status'])
n_fail = sum(1 for r in results_summary if 'FAILED' in r['status'])
print(f"   {n_ok} generated, {n_skip} skipped, {n_fail} failed.")
print(f"   Total wall time: {t_batch:.1f}s  ({t_batch/60:.1f} min)")
print(f"{'='*72}")



########################################################################
#  [1/8]  Profile: EZ17  (Illumina miSeq + Twist Bioscience)
#  n=136  alphabet=2mix_only(10)  N=100,000  M=50
########################################################################
   Profile: EZ17
   General: s=1.32000e-03  i=5.81000e-04  d=9.58000e-04  ld=2.33000e-04
   A: s=1.35000e-03  pi(ins)=5.90000e-04  d=9.90000e-04  ld=2.40000e-04  i(weight)=5.70000e-04
   C: s=1.35000e-03  pi(ins)=5.80000e-04  d=9.80000e-04  ld=2.30000e-04  i(weight)=5.90000e-04
   G: s=1.26000e-03  pi(ins)=5.70000e-04  d=9.40000e-04  ld=2.30000e-04  i(weight)=5.90000e-04
   T: s=1.32000e-03  pi(ins)=5.80000e-04  d=9.60000e-04  ld=2.30000e-04  i(weight)=5.80000e-04

GENERATING DATASET
   Samples       : 100,000
   Seq length    : 136
   Coverage depth: 50
   Alphabet      : 2mix_only (10 classes)
   Processed 10,000/100,000 | 180.6 samples/s | ETA 498.3s
   Processed 20,000/100,000 | 180.2 samples/s | ETA 444.0s
   Processed 30,000/


Read length statistics:
   Original length : 77
   Mean read length: 76.95
   Std  read length: 0.42
   Min  / Max      : 69 / 80


########################################################################
#  [4/8]  Profile: R21  (Oxford Nanopore MinION + Twist Bioscience)
#  n=136  alphabet=2mix_only(10)  N=100,000  M=50
########################################################################
   Profile: R21
   General: s=1.08000e-02  i=1.65000e-02  d=1.18000e-02  ld=3.36000e-03
   A: s=1.03900e-02  pi(ins)=1.58500e-02  d=1.19200e-02  ld=3.14000e-03  i(weight)=1.61000e-02
   C: s=1.04200e-02  pi(ins)=1.57800e-02  d=1.26000e-02  ld=3.34000e-03  i(weight)=1.63900e-02
   G: s=1.09400e-02  pi(ins)=1.70000e-02  d=1.26700e-02  ld=3.37000e-03  i(weight)=1.60400e-02
   T: s=1.13100e-02  pi(ins)=1.72900e-02  d=1.32700e-02  ld=3.57000e-03  i(weight)=1.73800e-02

GENERATING DATASET
   Samples       : 100,000
   Seq length    : 136
   Coverage depth: 50
   Alphabet      : 2mix_only (10 classes)
 


Read length statistics:
   Original length : 136
   Mean read length: 136.00
   Std  read length: 0.13
   Min  / Max      : 130 / 138


########################################################################
#  [7/8]  Profile: NP22  (Nanopore Pilot Nov-2022 + Twist Bioscience)
#  n=136  alphabet=2mix_only(10)  N=100,000  M=50
########################################################################
   Profile: NP22
   General: s=1.29000e-02  i=1.16000e-02  d=9.75000e-03  ld=2.82000e-03
   A: s=5.20000e-03  pi(ins)=5.82000e-03  d=3.73000e-03  ld=9.80000e-04  i(weight)=6.44000e-03
   C: s=1.49800e-02  pi(ins)=1.34000e-02  d=1.17800e-02  ld=3.36000e-03  i(weight)=1.33100e-02
   G: s=6.38000e-03  pi(ins)=6.67000e-03  d=4.74000e-03  ld=1.28000e-03  i(weight)=7.35000e-03
   T: s=2.43700e-02  pi(ins)=1.98300e-02  d=1.90300e-02  ld=5.50000e-03  i(weight)=1.86700e-02

GENERATING DATASET
   Samples       : 100,000
   Seq length    : 136
   Coverage depth: 50
   Alphabet      : 2mix_only (10 cla